In [ ]:
language = 'pt'

1. Gravação de Áudio Com Python (e Uma Pitada de JavaScript) 🎤

In [ ]:
# Referência: https://gist.github.com/korakot/c21c3476c024ad6d56d5f48b0bca92be

from IPython.display import Audio, display, Javascript
from google.colab import output
from base64 import b64decode

# Código JavaScript para gravar áudio do usuário usando a "MediaStream Recording API"
RECORD = """
const sleep  = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(e.srcElement.result)
  reader.readAsDataURL(blob)
})
var record = time => new Promise(async resolve => {
  stream = await navigator.mediaDevices.getUserMedia({ audio: true })
  recorder = new MediaRecorder(stream)
  chunks = []
  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()
  await sleep(time)
  recorder.onstop = async ()=>{
    blob = new Blob(chunks)
    text = await b2text(blob)
    resolve(text)
  }
  recorder.stop()
})
"""

def record(sec=5):
  # Executa o código JavaScript para gravar o áudio
  display(Javascript(RECORD))
  # Recebe o áudio gravado como resultado do JavaScript
  js_result = output.eval_js('record(%s)' % (sec * 1000))
   # Decodifica o áudio em base64
  audio = b64decode(js_result.split(',')[1])
  # Salva o áudio em um arquivo
  file_name = 'request_audio.wav'
  with open(file_name, 'wb') as f:
    f.write(audio)
  # Retorna o caminho do arquivo de áudio (pasta padrão do Google Colab)
  return f'/content/{file_name}'

# Grava o áudio do usuário por um tempo determinado (padrão 5 segundos)
print('Ouvindo...\n')
record_file = record()

# Exibe o áudio gravado
display(Audio(record_file, autoplay=False))

# 2. Reconhecimento de Fala 🧠

In [ ]:
# Instala a biblioteca necessária
!pip install -q -U google-generativeai

import google.generativeai as genai
import os

# Substitua 'TODO' pela sua chave de API do Google AI Studio
# Obtenha em: https://aistudio.google.com/app/apikey
os.environ['GOOGLE_API_KEY'] = 'TODO'

genai.configure(api_key=os.environ['GOOGLE_API_KEY'])

In [ ]:
# Configuração do modelo (Gemini 1.5 Flash é rápido e ótimo para áudio)
model = genai.GenerativeModel("gemini-2.5-flash")

# 1. Faz o upload do arquivo de áudio para a API do Gemini
# record_file deve ser o caminho do seu arquivo (ex: 'audio.wav')
audio_file = genai.upload_file(path=record_file)

# 2. Gera o conteúdo a partir do áudio
# Pedimos para ele responder ao áudio.
# Se você quiser apenas a transcrição, mude o prompt abaixo.
prompt = "Por favor, responda à pergunta ou comando contido neste áudio."
response = model.generate_content([prompt, audio_file])

# 3. Integração com a API do Gemini 💬

In [ ]:
# O Gemini processa o áudio nativamente, mas se você precisar do texto
# do que foi dito no áudio para outras partes do código:
transcription = model.generate_content(["Transcreva exatamente o que é dito neste áudio:", audio_file]).text

# Esta é a resposta do chat (o que seria o chatgpt_response)
chatgpt_response = response.text

print(f"Transcrição: {transcription}")
print(f"Resposta do Gemini: {chatgpt_response}")

# 4. Sintetizando a Resposta do ChatGPT Como Voz (gTTS) 🔊

In [ ]:
!pip install gTTS

In [ ]:
from gtts import  gTTS

# Cria um objeto gTTS com a resposta gerada pelo ChatGPT e a língua que será sintetizada em voz (variável "language").
gtts_object = gTTS(text=chatgpt_response, lang=language, slow=False)

# Salva o áudio da resposta no arquivo especificado (pasta padrão do Google Colab)
response_audio = "/content/response_audio.wav"
gtts_object.save(response_audio)

# Reproduz o áudio da resposta salvo no arquivo
display(Audio(response_audio, autoplay=True))